In [1]:
from pyspark.sql import SparkSession


In [2]:
spark = ( 
    SparkSession.builder
    .appName("OlistDataExploration")
    .master("local[*]")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/04 14:20:08 WARN Utils: Your hostname, pranav-ASUS-TUF-Gaming-A15-FA506ICB-FA506ICB, resolves to a loopback address: 127.0.1.1; using 192.168.0.157 instead (on interface wlp3s0)
26/10/04 14:20:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/04 14:20:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
print("spark started ")

spark started 


In [4]:
orders = spark.read.csv(
    "hdfs://localhost:9000/ecommerce/raw/olist/olist_orders_dataset.csv",
    header = True,
    inferSchema= True

)

In [5]:
customers = spark.read.csv(
    "hdfs://localhost:9000/ecommerce/raw/olist/olist_customers_dataset.csv",
    header = True,
    inferSchema= True

)

In [6]:
order_items = spark.read.csv(
    "hdfs://localhost:9000/ecommerce/raw/olist/olist_order_items_dataset.csv",
    header = True,
    inferSchema= True

)

In [7]:
products = spark.read.csv(
    "hdfs://localhost:9000/ecommerce/raw/olist/olist_products_dataset.csv",
    header = True,
    inferSchema= True

)

In [8]:
order_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



In [9]:
customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [10]:
products.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [11]:
print("orders count : ", orders.count())
print("Customers count : ", customers.count())
print("Orders Items count : ", order_items.count())
print("Products count : ", products.count())

orders count :  99441
Customers count :  99441
Orders Items count :  112650
Products count :  32951


In [12]:
orders.show(5, truncate = False)


+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |2017-10-02 11:07:15|2017-10-04 19:55:00         |2017-10-10 21:25:13          |2017-10-18 00:00:00          |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |2018-07-24 20:41:37     |2018-07-26 03:24:27|2018-07-26 14:3

In [13]:
order_items.show(5,truncate = False)


+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price|freight_value|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb4970a6e2683c13e61|48436dade18ac8b2bce089ec2a041202|2017-09-19 09:45:35|58.9 |13.29        |
|00018f77f2f0320c557190d7a144bdd3|1            |e5f2d52b802189ee658865ca93d83a8f|dd7ddc04e1b6c2c614352b383efe2d36|2017-05-03 11:05:13|239.9|19.93        |
|000229ec398224ef6ca0657da4fc703e|1            |c777355d18b72b67abbeef9df44fd0fd|5b51032eddd242adc84c38acab88f23d|2018-01-18 14:48:30|199.0|17.87        |
|00024acbcdf0a6daa1e931b038114c75|1            |7634da152a4610f1595efa

In [14]:
products.show(5,truncate = False)

+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff4541ed26657ea517e5|perfumaria           |40                 |287                       |1                 |225             |16               |10               |14              |
|3aa071139cb16b67ca9e5dea641aaa2f|artes                |44                 |276                       |1                 |1000            |30               |18               |20              |
|96bd76ec8810374ed1b65e291975717f|e

In [15]:
from pyspark.sql.functions import col, sum

In [16]:
order_items.select(
    "order_id",
    "product_id",
    "price",
    "freight_value"
    
).show(10)

+--------------------+--------------------+------+-------------+
|            order_id|          product_id| price|freight_value|
+--------------------+--------------------+------+-------------+
|00010242fe8c5a6d1...|4244733e06e7ecb49...|  58.9|        13.29|
|00018f77f2f0320c5...|e5f2d52b802189ee6...| 239.9|        19.93|
|000229ec398224ef6...|c777355d18b72b67a...| 199.0|        17.87|
|00024acbcdf0a6daa...|7634da152a4610f15...| 12.99|        12.79|
|00042b26cf59d7ce6...|ac6c3623068f30de0...| 199.9|        18.14|
|00048cc3ae777c65d...|ef92defde845ab845...|  21.9|        12.69|
|00054e8431b9d7675...|8d4f2bb7e93e6710a...|  19.9|        11.85|
|000576fe39319847c...|557d850972a7d6f79...| 810.0|        70.75|
|0005a1a1728c9d785...|310ae3c140ff94b03...|145.95|        11.65|
|0005f50442cb953dc...|4535b0e1091c278df...| 53.99|         11.4|
+--------------------+--------------------+------+-------------+
only showing top 10 rows


In [17]:
order_items.agg(
    sum("price").alias("total_sales")
).show()

+--------------------+
|         total_sales|
+--------------------+
|1.3591643699999392E7|
+--------------------+



In [18]:
orders_with_items = orders.join(
    order_items,
    on = "order_id",
    how = "inner"
)

In [19]:
orders_with_items.select(
    "order_id",
    "customer_id",
    "order_status",
    "product_id",
    "price",
    "freight_value"
).show(10,truncate=False)

+--------------------------------+--------------------------------+------------+--------------------------------+------+-------------+
|order_id                        |customer_id                     |order_status|product_id                      |price |freight_value|
+--------------------------------+--------------------------------+------------+--------------------------------+------+-------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |87285b34884572647811a353c7ac498a|29.99 |8.72         |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |595fac2a385ac33a80bd5114aec74eb8|118.7 |22.76        |
|47770eb9100c2d0c44946d9cf07ec65d|41ce2a54c0b03bf3443c3d931a367089|delivered   |aa4383b373c6aca5d8797843e5594415|159.9 |19.22        |
|949d5b44dbf5de918fe9c16f97b45f8a|f88197465ea7920adcdbec7375364d82|delivered   |d0b61bfb1de832b15ba9d266ca96e5b0|45.0  |27.2         |
|ad21c59c0840e6cb83a9ceb5573f8159|8ab97904e6daea8866dbd

In [20]:
print("Orders:", orders.count())
print("Order items:", order_items.count())
print("Joined rows:", orders_with_items.count())

Orders: 99441
Order items: 112650
Joined rows: 112650


In [21]:
orders_with_products= orders_with_items.join(
    products,
    on = "product_id",
    how = "left"
)

In [22]:
orders_with_products.select(
    "order_id",
    "customer_id",
    "product_id",
    "product_category_name",
    "price",
).show(5,truncate=True)

+--------------------+--------------------+--------------------+---------------------+-----+
|            order_id|         customer_id|          product_id|product_category_name|price|
+--------------------+--------------------+--------------------+---------------------+-----+
|00010242fe8c5a6d1...|3ce436f183e68e078...|4244733e06e7ecb49...|           cool_stuff| 58.9|
|00018f77f2f0320c5...|f6dd3ec061db4e398...|e5f2d52b802189ee6...|             pet_shop|239.9|
|000229ec398224ef6...|6489ae5e4333f3693...|c777355d18b72b67a...|     moveis_decoracao|199.0|
|00024acbcdf0a6daa...|d4eb9395c8c0431ee...|7634da152a4610f15...|           perfumaria|12.99|
|00042b26cf59d7ce6...|58dbd0b2d70206bf4...|ac6c3623068f30de0...|   ferramentas_jardim|199.9|
+--------------------+--------------------+--------------------+---------------------+-----+
only showing top 5 rows


In [23]:
print(orders_with_products.first())

Row(product_id='c8e7c2ef329fcda4a233e7e2f8bb8b7d', order_id='00310b0c75bb13015ec4d82d341865a4', customer_id='0dad07848c618cc5a4679a1bfe1db8d2', order_status='canceled', order_purchase_timestamp=datetime.datetime(2018, 8, 15, 14, 29, 8), order_approved_at=datetime.datetime(2018, 8, 15, 15, 4, 25), order_delivered_carrier_date=None, order_delivered_customer_date=None, order_estimated_delivery_date=datetime.datetime(2018, 8, 29, 0, 0), order_item_id=1, seller_id='a2deecd5398f5df4987110c80a1972a3', shipping_limit_date=datetime.datetime(2018, 8, 17, 15, 4, 25), price=39.9, freight_value=15.38, product_category_name='utilidades_domesticas', product_name_lenght=58, product_description_lenght=1245, product_photos_qty=1, product_weight_g=200, product_length_cm=16, product_height_cm=12, product_width_cm=11)


In [24]:
print("Orders:", orders.count())
print("Order items:", order_items.count())
print("Orders with products:", orders_with_products.count())

Orders: 99441
Order items: 112650
Orders with products: 112650


In [25]:
orders_with_customers = orders_with_products.join(
    customers,
    on = "customer_id",
    how = "left"
)

In [26]:
print(orders_with_customers.count())
print(orders_with_customers.first())

112650


26/10/04 14:20:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Row(customer_id='0dad07848c618cc5a4679a1bfe1db8d2', product_id='c8e7c2ef329fcda4a233e7e2f8bb8b7d', order_id='00310b0c75bb13015ec4d82d341865a4', order_status='canceled', order_purchase_timestamp=datetime.datetime(2018, 8, 15, 14, 29, 8), order_approved_at=datetime.datetime(2018, 8, 15, 15, 4, 25), order_delivered_carrier_date=None, order_delivered_customer_date=None, order_estimated_delivery_date=datetime.datetime(2018, 8, 29, 0, 0), order_item_id=1, seller_id='a2deecd5398f5df4987110c80a1972a3', shipping_limit_date=datetime.datetime(2018, 8, 17, 15, 4, 25), price=39.9, freight_value=15.38, product_category_name='utilidades_domesticas', product_name_lenght=58, product_description_lenght=1245, product_photos_qty=1, product_weight_g=200, product_length_cm=16, product_height_cm=12, product_width_cm=11, customer_unique_id='ec979208947bbba310f2ad8e50963725', customer_zip_code_prefix=31160, customer_city='belo horizonte', customer_state='MG')


In [27]:
from pyspark.sql.functions import col

orders_with_customers = orders_with_customers.withColumn(
    "total_item_cost",
    col("price")+ col("freight_value")
)

In [28]:
from pyspark.sql.functions import sum, avg, count

orders_with_customers.agg(
    sum("price").alias("total_product_revenue"),
    sum("freight_value").alias("total_freight"),
    sum("total_item_cost").alias("total_revenue_with_freight"),
    avg("price").alias("average_item_price"),
    count("*").alias("total_items")
).show()

+---------------------+------------------+--------------------------+------------------+-----------+
|total_product_revenue|     total_freight|total_revenue_with_freight|average_item_price|total_items|
+---------------------+------------------+--------------------------+------------------+-----------+
| 1.3591643699999392E7|2251909.5399999325|      1.5843553239999713E7|120.65373901464174|     112650|
+---------------------+------------------+--------------------------+------------------+-----------+



In [29]:
orders.groupBy("order_status").count().orderBy("count", ascending=False).show()

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



In [30]:
delivered_orders = orders_with_customers.filter(
    col("order_status")=="delivered"
)

In [31]:
print("Delivered orders:", delivered_orders.select("order_id").distinct().count())
print("Delivered order items:", delivered_orders.count())

Delivered orders: 96478
Delivered order items: 110197


In [32]:
delivered_orders.agg(
    sum("price").alias("product_revenue"),
    sum("freight_value").alias("freight_revenue"),
    sum("total_item_cost").alias("total_value")
).show()

+--------------------+-----------------+-------------------+
|     product_revenue|  freight_revenue|        total_value|
+--------------------+-----------------+-------------------+
|1.3221498109999347E7|2198275.639999931|1.541977374999974E7|
+--------------------+-----------------+-------------------+



In [33]:
from pyspark.sql.functions import countDistinct
customer_revenue = (
    delivered_orders
    .groupBy("customer_unique_id")
    .agg(
        sum("price").alias("total_spent"),
        countDistinct("order_id").alias("number_of_orders")
    )
    .orderBy("total_spent", ascending=False)
)

In [34]:
print("Unique customers:", customer_revenue.count())

Unique customers: 93358


In [35]:
print(customer_revenue.first())

Row(customer_unique_id='0a0a92112bd4c708ca5fde585afaa872', total_spent=13440.0, number_of_orders=1)


In [36]:
from pyspark.sql.functions import sum , countDistinct , round

customer_revenue = (
    delivered_orders
    .groupBy("customer_unique_id")
    .agg(
        sum("price").alias("total_spent"),
        countDistinct("order_id").alias("number_of_orders")
    )
    .withColumn(
        "average_order_value",
        round(col("total_spent")/ col("number_of_orders"),2)
    )
    .orderBy("total_spent", ascending= False)
)

In [37]:
print(customer_revenue.first())

Row(customer_unique_id='0a0a92112bd4c708ca5fde585afaa872', total_spent=13440.0, number_of_orders=1, average_order_value=13440.0)


In [38]:
product_revenue = (
    delivered_orders
    .groupBy("product_id")
    .agg(
        sum("price").alias("product_revenue"),
        count("*").alias("units_sold")
    )
    .orderBy("product_revenue", ascending=False)
)

In [39]:
print("Products sold:", product_revenue.count())

Products sold: 32216


In [40]:
print(product_revenue.first())

Row(product_id='bb50f2e236e5eea0100680137654686c', product_revenue=63560.0, units_sold=194)


In [1]:
print("Hello")

Hello


In [41]:
from pyspark.sql import SparkSession

print("Python is working")

Python is working


In [42]:
print("Notebook is responsive")

Notebook is responsive


In [43]:
from pyspark.sql.functions import col

delivered_orders = orders.filter(
    col("order_status") == "delivered"
)

print("delivered_orders created")

delivered_orders created


In [44]:
print(orders.count())
print(customers.count())
print(order_items.count())
print(products.count())

99441
99441
112650
32951


In [45]:
from pyspark.sql.functions import col

delivered_orders = orders.filter(
    col("order_status") == "delivered"
)

print(delivered_orders.count())

96478


In [46]:
from pyspark.sql.functions import col

delivered_orders = orders.filter(
    col("order_status") == "delivered"
)

print("Delivered orders:", delivered_orders.count())

orders_with_items = delivered_orders.join(
    order_items,
    on="order_id",
    how="inner"
)

print("After order_items join:", orders_with_items.count())

orders_with_products = orders_with_items.join(
    products,
    on="product_id",
    how="left"
)

print("After products join:", orders_with_products.count())

orders_with_customers = orders_with_products.join(
    customers,
    on="customer_id",
    how="left"
)

print("After customers join:", orders_with_customers.count())

Delivered orders: 96478
After order_items join: 110197
After products join: 110197
After customers join: 110197


In [47]:
print("Rows:", orders_with_customers.count())

business_metrics = orders_with_customers.agg(
    sum("price").alias("product_revenue"),
    sum("freight_value").alias("freight"),
    sum(col("price") + col("freight_value")).alias("total_value")
)

business_metrics.show()

Rows: 110197
+--------------------+-----------------+-------------------+
|     product_revenue|          freight|        total_value|
+--------------------+-----------------+-------------------+
|1.3221498109999347E7|2198275.639999931|1.541977374999974E7|
+--------------------+-----------------+-------------------+



In [49]:
from pyspark.sql.functions import col, sum

test_df = orders_with_customers.withColumn(
    "total_item_cost",
    col("price") + col("freight_value")
)

test_df.agg(
    sum("price").alias("product_revenue"),
    sum("freight_value").alias("freight"),
    sum("total_item_cost").alias("total_value")
).show()

+--------------------+-----------------+-------------------+
|     product_revenue|          freight|        total_value|
+--------------------+-----------------+-------------------+
|1.3221498109999347E7|2198275.639999931|1.541977374999974E7|
+--------------------+-----------------+-------------------+



In [50]:
test_df.select(
    "price",
    "freight_value",
    "total_item_cost"
).show(10)

+------+-------------+------------------+
| price|freight_value|   total_item_cost|
+------+-------------+------------------+
|  58.9|        13.29|             72.19|
| 239.9|        19.93|            259.83|
| 199.0|        17.87|            216.87|
| 12.99|        12.79|             25.78|
| 199.9|        18.14|218.04000000000002|
|  21.9|        12.69|34.589999999999996|
|  19.9|        11.85|             31.75|
| 810.0|        70.75|            880.75|
|145.95|        11.65|             157.6|
| 53.99|         11.4|             65.39|
+------+-------------+------------------+
only showing top 10 rows


In [4]:
processed_orders = spark.read.parquet(
    "hdfs://localhost:9000/ecommerce/processed/olist"
)

print("Rows:", processed_orders.count())

processed_orders.select(
    "price",
    "freight_value",
    "total_item_cost"
).show(10)

NameError: name 'spark' is not defined